# 02 - LangChain RAG Pipeline

Build a Retrieval-Augmented Generation pipeline using **Weaviate** as the vector store and the **LiteLLM gateway** as the unified LLM front door. LangChain's `ChatOpenAI` and `OpenAIEmbeddings` are pointed at LiteLLM, which routes to whichever upstream (Ollama/OpenAI/Anthropic/…) is configured.

In [ ]:
import os
from dotenv import load_dotenv
import weaviate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

## 1. Connect to services

In [ ]:
# Weaviate (vector store)
weaviate_url = os.getenv("WEAVIATE_URL").replace("http://", "")
host, port = weaviate_url.split(":")
wv_client = weaviate.connect_to_custom(
    http_host=host,
    http_port=int(port),
    http_secure=False,
    grpc_host=host,
    grpc_port=50051,
    grpc_secure=False,
)

# LiteLLM gateway — point LangChain's OpenAI clients at it.
litellm_base = os.environ["OPENAI_API_BASE"]   # http://litellm:4000/v1
litellm_key = os.environ["OPENAI_API_KEY"]

chat_model = os.getenv("LITELLM_DEFAULT_MODEL", "ollama/qwen3.6:latest")
embed_model = os.getenv("LITELLM_EMBEDDING_MODEL", "ollama/nomic-embed-text")

llm = ChatOpenAI(model=chat_model, base_url=litellm_base, api_key=litellm_key)
embeddings = OpenAIEmbeddings(model=embed_model, base_url=litellm_base, api_key=litellm_key)

print(f"✅ Weaviate ready: {wv_client.is_ready()}")
print(f"✅ LiteLLM chat model: {chat_model}")
print(f"✅ LiteLLM embedding model: {embed_model}")

## 2. Sample documents

In [ ]:
documents = [
    "Atlas is a self-hosted engineering platform for gen-AI, ML, and data work.",
    "LiteLLM is a unified OpenAI-compatible gateway for every LLM provider.",
    "Weaviate is a vector database optimized for semantic search.",
    "JupyterHub enables interactive data science workflows.",
    "Neo4j stores data as graphs with nodes and relationships.",
]

print(f"Sample documents: {len(documents)}")

## 3. Embed and store in Weaviate

In [ ]:
from weaviate.classes.config import Configure, Property, DataType

if not wv_client.collections.exists("Document"):
    wv_client.collections.create(
        name="Document",
        properties=[Property(name="content", data_type=DataType.TEXT)],
    )

collection = wv_client.collections.get("Document")

vectors = embeddings.embed_documents(documents)
for doc, vec in zip(documents, vectors):
    collection.data.insert(properties={"content": doc}, vector=vec)

print("✅ Documents stored in Weaviate")

## 4. RAG query

In [ ]:
query = "What is a vector database?"
query_vec = embeddings.embed_query(query)

results = collection.query.near_vector(near_vector=query_vec, limit=2)
context = "\n".join(obj.properties["content"] for obj in results.objects)

prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
answer = llm.invoke(prompt)

print(f"Query: {query}")
print(f"\nContext:\n{context}")
print(f"\nAnswer: {answer.content}")